In [39]:
from SteerEnergyStorage.Formulations.ElectrodeFormulations import ElectrodeFormulation
from SteerEnergyStorage.Constructions.Electrodes import Cathode, Anode
from SteerEnergyStorage.Formulations.ElectrodeAssemblies import Stack
from SteerEnergyStorage.Constructions.Cells import StackedPouchCell
from SteerEnergyStorage.Materials.ElectrodeMaterials import CathodeMaterial, AnodeMaterial, Binder, ConductiveAdditive
from SteerEnergyStorage.Materials.CurrentCollectors import CurrentCollector
from SteerEnergyStorage.Materials.Separators import Separator
from SteerEnergyStorage.Materials.Electrolytes import Electrolyte
from SteerEnergyStorage.Constructions.Containers import Pouch
from SteerEnergyStorage.Materials.other import Laminate, Tape, Terminal

In [40]:
import plotly.express as px
import pandas as pd

In [41]:
CathodeMaterial.get_available_materials()

['lfp_2',
 'nvp',
 'suxiang_xn33s',
 'pb_powder',
 'nvpf',
 'ndi_synthetic',
 'nmc622',
 'faradion_gen2_4.35v',
 'm2-c',
 'lfp',
 'faradion_gen2_4.25v',
 'faradion_gen2_4.1v',
 'nfpp',
 'prussian_white',
 'standard_potential_sodacam-001']

In [42]:
AnodeMaterial.get_available_materials()

['pgt_synthetic',
 'faradion_hc',
 'tin_powder_dgme',
 'faradion_hc_commercial',
 'pb_powder',
 'longtime_shc300',
 'synthetic_graphite',
 'longtime_shc300-h1',
 'kuraray_kuranode_type_i',
 'na_hardcarbon']

In [43]:
cathode_active_material = CathodeMaterial(name="ndi_synthetic", specific_cost=2, density=1.2)
anode_active_material = AnodeMaterial(name="pgt_synthetic", specific_cost=2, density=1.2)


In [44]:
cathode_formulation = ElectrodeFormulation(active_materials={cathode_active_material: 100})
anode_formulation = ElectrodeFormulation(active_materials={anode_active_material: 100})

In [45]:
# construct the cathode 

cathode_current_collector = CurrentCollector(formula="Al", 
                                             thickness=15, 
                                             length=16.0,
                                             width=10.8,
                                             bare_area=8.22)

cathode = Cathode(formulation=cathode_formulation,
                  mass_loading=100,
                  current_collector=cathode_current_collector,
                  calender_density=1.2)


In [46]:
# construct the anode 

anode_current_collector = CurrentCollector(formula="Al",
                                           thickness=15,
                                           length=16.0,
                                           width=10.8,
                                           bare_area=7.55)

anode = Anode(formulation=anode_formulation,
              mass_loading=150,
              current_collector=anode_current_collector,
              calender_density=1.2)


In [47]:
# construct seperator
separator = Separator(thickness=100, 
                      areal_cost=0.3, 
                      density=0.5, 
                      width=11.0, 
                      fold_length=18.6,
                      porosity=47)

# construct the stack
stack = Stack(anode=anode, 
              cathode=cathode,
              separator=separator,
              n_layers=26,
              name = "Stack")


In [48]:
# make electrolyte
electrolyte = Electrolyte(specific_cost=0.01, density=1)

# make the pouch
laminate = Laminate(thickness=113, 
                    areal_mass=18, 
                    areal_cost=4.64)

tape = Tape(mass=0.3)

# Make the cell
pos_terminal = Terminal(mass = 1, specific_cost = 16, name="Positive Terminal")
neg_terminal = Terminal(mass = 1, specific_cost = 16, name="Negative Terminal")

pouch = Pouch(laminate=laminate, 
              heat_seal_size_sides=7, 
              heat_seal_size_top=22, 
              tape=tape,
              positive_terminal=pos_terminal,
              negative_terminal=neg_terminal)

cell = StackedPouchCell(pouch=pouch,
                        stack=stack,
                        electrolyte=electrolyte,
                        electrolyte_overfill=10,
                        reversible_capacity=30.934,
                        irreversible_capacity=1.215)

In [49]:
cell._full_cell_curve = cell._full_cell_curve.assign(voltage = lambda x: x['voltage'] + 0.5)


In [ ]:
figure = cell.get_capacity_voltage_plot(width=900, height=600)
figure.show()

In [ ]:
print(f"The energy the cell can hold is {cell.energy} Wh")
print(f"The specific energy of the cell is {cell.specific_energy} Wh/kg")
print(f"The cost of the cell is {cell.cost} $")
print(f"The normalized cost of the cell is {cell.normalized_cost} $/kWh")
print(f"The energy density of the cell is {cell.energy_density} Wh/L")

The energy the cell can hold is 16.17 Wh
The specific energy of the cell is 6.54 Wh/kg
The cost of the cell is 5.21 $
The normalized cost of the cell is 322.52 $/kWh
The energy density of the cell is 5.32 Wh/L


In [ ]:
figure = cell.get_cost_breakdown_plot(width=500, height=500)
figure.show() 

In [ ]:
figure = cell.get_mass_breakdown_plot(width=800, height=400)
figure.show()